<a href="https://colab.research.google.com/github/Ravindra1972/Anaytics-in-finance-using-Python/blob/main/Bond_Pricer_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import numpy as np
from scipy.optimize import brentq

def bond_price(face, coupon_rate, ytm, n_years, freq=2):
    """
    Price a bond using discounted cash flows.
    :param face: Face (par) value
    :param coupon_rate: Annual coupon rate (e.0.07)
    :param ytm: Yield to maturity (annual)
    :param n_years: Years to maturity
    :param freq: Coupons per year (default=2)
    """
    c = coupon_rate * face / freq  # Coupon / period
    y = ytm / freq  # Yield / period
    n = int(n_years * freq)  # Total periods

    # PV of coupons (annuity formula)
    if y == 0:
        pv_coupons = c * n
    else:
        pv_coupons = c * (1 - (1 + y)**(-n)) / y

    # PV of face value
    pv_face = face / (1 + y)**n

    return pv_coupons + pv_face

def ytm_solve(price, face, coupon_rate, n_years, freq=2):
    """Solve for YTM given market price."""
    def objective(ytm):
        return bond_price(face, coupon_rate, ytm, n_years, freq) - price
    return brentq(objective, -0.1, 2.0)

In [8]:
def macaulay_duration(face, coupon_rate, ytm, n_years, freq=2):
    """Compute Macaulay duration."""
    c = coupon_rate * face / freq
    y = ytm / freq
    n = int(n_years * freq)
    price = bond_price(face, coupon_rate, ytm, n_years, freq)

    weighted_sum = 0
    for t in range(1, n + 1):
        cf = c if t < n else c + face
        pv = cf / (1 + y)**t
        weighted_sum += (t / freq) * pv
    return weighted_sum / price

def dv01(face, coupon_rate, ytm, n_years, freq=2):
    """Dollar value of 1 basis point."""
    p_up = bond_price(face, coupon_rate, ytm + 0.0001, n_years, freq)
    p_dn = bond_price(face, coupon_rate, ytm - 0.0001, n_years, freq)
    return (p_dn - p_up) / 2

In [13]:
# === DEMONSTRATIONS ===
print ("=" * 55)
print (" BOND PRICING DEMO ")
print ("=" * 55)
 # 1. Basic bond pricing
params = {'face': 100 , 'coupon_rate': 0.07 ,
'n_years': 10, 'freq': 2}
print ("\n--- Price at Different Yields ---")
for y in [0.05 , 0.06 , 0.07 , 0.08 , 0.09]:
 p = bond_price (** params , ytm =y)
 status = " Premium " if p > 100 else \
 (" Discount " if p < 100 else "At Par")
 print (f" y={y :.0%}: Price =${p:.4f} ({ status })")

 BOND PRICING DEMO 

--- Price at Different Yields ---
 y=5%: Price =$115.5892 ( Premium )
 y=6%: Price =$107.4387 ( Premium )
 y=7%: Price =$100.0000 (At Par)
 y=8%: Price =$93.2048 ( Discount )
 y=9%: Price =$86.9921 ( Discount )


In [14]:
# 2. YTM calculation
market_price = 104.21
solved_ytm = ytm_solve ( market_price , ** params )
print (f"\n--- YTM Solver ---")
print (f" Market price : ${ market_price :.2f}")
print (f" Solved YTM: { solved_ytm :.4%} ")



--- YTM Solver ---
 Market price : $104.21
 Solved YTM: 6.4229% 


In [16]:
# 3. Duration
print (f"\n--- Duration and DV01 ---")
for mat in [2, 5, 10, 20, 30]:
 dur = macaulay_duration (100 , 0.07 , 0.07 , mat)
 d01 = dv01 (100 , 0.07 , 0.07 , mat)
print (f" {mat :2d}yr: Duration ={ dur :.2f}yr , "
f" DV01 =${d01 :.4f}")


--- Duration and DV01 ---
 30yr: Duration =12.91yr ,  DV01 =$0.1247


In [17]:
# 4. Zero - coupon bond
print (f"\n--- Zero - Coupon Bond ---")
for mat in [1, 5, 10, 30]:
 p = bond_price (100 , 0.0 , 0.06 , mat , freq =1)
 print (f" {mat :2d}yr ZCB @ 6%: ${p:.2f} "
 f"( discount ={100 -p:.2f})")


--- Zero - Coupon Bond ---
  1yr ZCB @ 6%: $94.34 ( discount =5.66)
  5yr ZCB @ 6%: $74.73 ( discount =25.27)
 10yr ZCB @ 6%: $55.84 ( discount =44.16)
 30yr ZCB @ 6%: $17.41 ( discount =82.59)


In [20]:
# 5. Credit spread analysis
print (f"\n--- Credit Spread Example ---")
govt_yield = 0.070
corp_yield = 0.082
spread_bps = ( corp_yield - govt_yield ) * 10000
govt_price = bond_price (100 , 0.07 , govt_yield , 10)
corp_price = bond_price (100 , 0.07 , corp_yield , 10)
print (f" Govt bond (7.0%) : ${ govt_price :.2f}")
print (f" Corp bond (8.2%) : ${ corp_price :.2f}")
print (f" Credit spread : { spread_bps :.0f} bps ")
print (f" Price difference : ${ govt_price - corp_price :.2f}")


--- Credit Spread Example ---
 Govt bond (7.0%) : $100.00
 Corp bond (8.2%) : $91.92
 Credit spread : 120 bps 
 Price difference : $8.08
